[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/dlt-certified/blob/main/notebooks/day-07-quality.ipynb)

---
# Day 7 · Error Handling and Data Quality
**certified-journeys / dlt-certified** &nbsp;|&nbsp; Data Quality

> **Goal for today:** Build pipelines that fail loudly on bad data or recover gracefully. Combine dlt's built-in quality controls with custom filtering and Pydantic validation so bad records never silently corrupt your warehouse.

---
## Why data quality matters in pipelines

Pipelines fail silently more than they fail loudly. A missing field becomes `NULL`. A wrong type gets coerced to something unexpected. Suddenly your dashboards are wrong and nobody knows why.

dlt gives you several layers of defence:

| Layer | Mechanism | When it fires |
|---|---|---|
| **Type coercion** | dlt tries to cast values to the inferred type | Every row, automatically |
| **Schema contracts** | `freeze` raises on unexpected columns/types | Schema evolution events |
| **Custom filtering** | Your generator skips bad records | Before dlt sees the data |
| **Pydantic validation** | Model validation raises on bad records | Before dlt sees the data |
| **Pipeline exceptions** | `pipeline.run()` raises on load errors | During the run |
| **`last_trace`** | Post-run inspection of what happened | After every run |

> **Tip:** Use `fail_map_on_single_error=False` during development to keep going past bad records. Flip it to `True` in production for strict quality.

---
## Step 1 · Install dlt

In [ ]:
%pip install -q "dlt[duckdb]" pydantic

---
## Step 2 · Default behaviour: dlt coerces bad data

Let's see what dlt does with intentionally messy data before we add any quality controls.

In [ ]:
import dlt
import logging

# A resource with intentionally bad data
@dlt.resource(write_disposition="replace")
def messy_orders():
    yield [
        # Good record
        {"order_id": 1, "customer_id": "CUST01", "amount": 99.99, "quantity": 2},
        # amount is a string that looks like a number — dlt will coerce
        {"order_id": 2, "customer_id": "CUST02", "amount": "149.99", "quantity": 1},
        # quantity is None — becomes NULL in the table
        {"order_id": 3, "customer_id": "CUST03", "amount": 49.99,  "quantity": None},
        # Extra unexpected column — dlt will add it to the schema
        {"order_id": 4, "customer_id": "CUST04", "amount": 29.99,  "quantity": 3,
         "promo_code": "SAVE10"},  # new column, not in original schema
        # amount is a non-numeric string — dlt will fail on this one
        {"order_id": 5, "customer_id": "CUST05", "amount": "not_a_number", "quantity": 1},
    ]

pipeline = dlt.pipeline(
    pipeline_name="quality_demo",
    destination="duckdb",
    dataset_name="raw",
)

# Run without any quality controls — watch what happens
try:
    info = pipeline.run(messy_orders())
    print("Pipeline succeeded (with possible coercions):")
    print(info)
except Exception as e:
    print(f"Pipeline raised an exception: {type(e).__name__}: {e}")

In [ ]:
# Inspect what actually landed — look at the types and NULL values
with pipeline.sql_client() as client:
    with client.execute_query("SELECT * FROM raw.messy_orders ORDER BY order_id") as cur:
        df = cur.df()
        print(df.to_string(index=False))
        print("\nDtypes:", dict(df.dtypes))

**What just happened?**
- `"149.99"` (string) → coerced to float. dlt is lenient by default.
- `None` → became a SQL `NULL` in the quantity column
- `"promo_code"` → dlt *evolved the schema* and added the column automatically
- `"not_a_number"` → dlt likely failed or coerced to NULL depending on schema state

This is dlt's default: permissive. Good for development, risky for production.

---
## Step 3 · Schema contracts: freeze evolution

Schema contracts let you control what happens when dlt encounters unexpected data.

In [ ]:
# Schema contract modes:
# evolve  (default) — add new columns, evolve types freely
# freeze            — raise an exception on any schema change
# discard_rows      — drop rows that would cause schema changes
# discard_value     — drop only the unexpected column, keep the row

@dlt.resource(
    write_disposition="replace",
    schema_contract="freeze",  # any new column = exception
)
def strict_orders():
    # First two records are fine
    yield {"order_id": 1, "amount": 99.99, "status": "completed"}
    yield {"order_id": 2, "amount": 49.99, "status": "pending"}
    # This record has a new column — will trigger schema contract violation
    yield {"order_id": 3, "amount": 29.99, "status": "completed",
           "unexpected_field": "oops"}  # <-- new column not in schema

strict_pipeline = dlt.pipeline(
    pipeline_name="strict_quality",
    destination="duckdb",
    dataset_name="strict",
)

# First run: establishes the schema with 3 columns
info_first = strict_pipeline.run(
    [{"order_id": 1, "amount": 99.99, "status": "completed"},
     {"order_id": 2, "amount": 49.99, "status": "pending"}],
    table_name="orders",
    write_disposition="replace",
)
print("First run (establishes schema):")
print(info_first)

In [ ]:
# Second run: try to add a new column — freeze should raise
try:
    info_second = strict_pipeline.run(
        strict_orders(),
    )
    print("Run succeeded — freeze may not have triggered on new resource")
    print(info_second)
except Exception as e:
    print(f"Schema contract violation caught:")
    print(f"  {type(e).__name__}: {e}")
    print("\nThis is the expected behaviour in production — fail loudly on schema drift.")

**Schema contract summary:**

| Mode | New column | Type mismatch | Use when |
|---|---|---|---|
| `evolve` | Added silently | Coerced | Development / exploration |
| `freeze` | Exception raised | Exception raised | Production — strict contracts |
| `discard_rows` | Row dropped | Row dropped | Tolerant production pipelines |
| `discard_value` | Column dropped from row | Value dropped | Keeping rows, ignoring bad fields |

---
## Step 4 · Custom filtering in the generator

The cleanest quality control is filtering *before* dlt sees the data — in the generator itself. Bad records never even reach the normaliser.

In [ ]:
import logging

# Set up a simple logger so we can see what gets skipped
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("quality_pipeline")

# Raw records from an upstream source — mix of good and bad
RAW_RECORDS = [
    {"order_id": 1,    "customer_id": "CUST01", "amount": 99.99,  "quantity": 2},
    {"order_id": None, "customer_id": "CUST02", "amount": 49.99,  "quantity": 1},   # missing PK
    {"order_id": 3,    "customer_id": None,      "amount": 79.99,  "quantity": 1},   # missing customer
    {"order_id": 4,    "customer_id": "CUST04", "amount": -5.00,  "quantity": 1},   # negative amount
    {"order_id": 5,    "customer_id": "CUST05", "amount": 199.99, "quantity": 0},   # zero quantity
    {"order_id": 6,    "customer_id": "CUST06", "amount": 29.99,  "quantity": 3},   # good
    {"order_id": 7,    "customer_id": "CUST07", "amount": 0.0,    "quantity": 1},   # zero amount (warn)
    {"order_id": 8,    "customer_id": "CUST08", "amount": 149.99, "quantity": 2},   # good
]

@dlt.resource(primary_key="order_id", write_disposition="replace")
def validated_orders():
    """
    Yields only valid orders. Logs and skips bad records.
    This is the first line of defence — runs before dlt normalises anything.
    """
    skipped = 0
    loaded = 0

    for record in RAW_RECORDS:
        order_id = record.get("order_id")

        # Rule 1: order_id must be present and non-null
        if order_id is None:
            logger.warning(f"SKIP: missing order_id in record: {record}")
            skipped += 1
            continue

        # Rule 2: customer_id must be present
        if not record.get("customer_id"):
            logger.warning(f"SKIP order_id={order_id}: missing customer_id")
            skipped += 1
            continue

        # Rule 3: amount must be positive
        if record.get("amount", 0) <= 0:
            logger.warning(f"SKIP order_id={order_id}: invalid amount={record.get('amount')}")
            skipped += 1
            continue

        # Rule 4: quantity must be > 0
        if record.get("quantity", 0) <= 0:
            logger.warning(f"SKIP order_id={order_id}: invalid quantity={record.get('quantity')}")
            skipped += 1
            continue

        # Record passed all checks — yield it to dlt
        loaded += 1
        yield record

    logger.info(f"Generator complete: {loaded} loaded, {skipped} skipped")


filter_pipeline = dlt.pipeline(
    pipeline_name="filtered_quality",
    destination="duckdb",
    dataset_name="clean",
)

info = filter_pipeline.run(validated_orders())
print("\nLoad info:")
print(info)

In [ ]:
# Only valid records should be in the table
with filter_pipeline.sql_client() as client:
    with client.execute_query("SELECT order_id, customer_id, amount, quantity FROM clean.validated_orders ORDER BY order_id") as cur:
        print("Records that passed quality checks:")
        print(cur.df().to_string(index=False))

**What just happened?**
- 4 records were skipped before dlt saw them (logged as WARNINGs)
- Only 4 valid records landed in the table
- The generator keeps a `skipped` counter — you could emit this as a metric
- dlt never saw the bad records — no schema coercion, no silent NULLs

---
## Step 5 · Pydantic validation

For complex schemas, Pydantic is cleaner than manual checks. Define your expected shape once as a model — validation is automatic and gives you detailed error messages.

In [ ]:
from pydantic import BaseModel, Field, field_validator, ValidationError
from typing import Optional, Literal

# Define the expected shape of a valid order
class Order(BaseModel):
    order_id:    int
    customer_id: str
    amount:      float = Field(gt=0, description="Must be positive")
    quantity:    int   = Field(gt=0, description="Must be at least 1")
    status:      Literal["completed", "pending", "refunded"] = "pending"
    currency:    str   = "USD"

    @field_validator("customer_id")
    @classmethod
    def customer_id_format(cls, v: str) -> str:
        """Customer IDs must start with 'CUST'."""
        if not v.startswith("CUST"):
            raise ValueError(f"customer_id must start with 'CUST', got: {v!r}")
        return v

    @field_validator("amount", mode="before")
    @classmethod
    def coerce_amount(cls, v):
        """Try to parse string amounts like '49.99'."""
        try:
            return float(v)
        except (TypeError, ValueError):
            raise ValueError(f"Cannot convert amount to float: {v!r}")


# Raw data with various problems
RAW_ORDERS_V2 = [
    {"order_id": 1, "customer_id": "CUST01", "amount": 99.99, "quantity": 2, "status": "completed"},
    {"order_id": 2, "customer_id": "CUST02", "amount": "49.99", "quantity": 1},          # string amount — coerced
    {"order_id": 3, "customer_id": "user_123", "amount": 29.99, "quantity": 1},          # bad customer_id format
    {"order_id": 4, "customer_id": "CUST04", "amount": -10.00, "quantity": 1},           # negative amount
    {"order_id": 5, "customer_id": "CUST05", "amount": 79.99,  "quantity": 2, "status": "shipped"},  # invalid status
    {"order_id": 6, "customer_id": "CUST06", "amount": 199.99, "quantity": 3, "status": "refunded"},  # good
]


@dlt.resource(primary_key="order_id", write_disposition="replace")
def pydantic_validated_orders():
    """
    Validates each record against the Order Pydantic model.
    Bad records are logged and skipped; good records are yielded.
    """
    for raw in RAW_ORDERS_V2:
        try:
            # Validate — raises ValidationError if anything is wrong
            order = Order.model_validate(raw)
            # Yield the validated dict — Pydantic has coerced all types
            yield order.model_dump()
        except ValidationError as e:
            # Log the validation errors clearly
            errors = [{"field": err["loc"], "msg": err["msg"]} for err in e.errors()]
            logger.warning(f"VALIDATION FAILED for order_id={raw.get('order_id')}: {errors}")


pydantic_pipeline = dlt.pipeline(
    pipeline_name="pydantic_quality",
    destination="duckdb",
    dataset_name="validated",
)

info = pydantic_pipeline.run(pydantic_validated_orders())
print("\nLoad info:")
print(info)

In [ ]:
# Inspect what passed validation
with pydantic_pipeline.sql_client() as client:
    with client.execute_query(
        "SELECT order_id, customer_id, amount, quantity, status, currency FROM validated.pydantic_validated_orders ORDER BY order_id"
    ) as cur:
        print("Records that passed Pydantic validation:")
        print(cur.df().to_string(index=False))

**What just happened?**
- Pydantic's `model_validate()` checked every field against its type and constraints
- `"49.99"` was coerced to `49.99` (float) by the `@field_validator`
- Records 3, 4, and 5 failed and were logged as warnings
- `order.model_dump()` returns a clean Python dict — what dlt receives has guaranteed types

---
## Step 6 · Catching pipeline errors with try/except

In [ ]:
import dlt
from dlt.common.exceptions import PipelineException

@dlt.resource
def bad_resource():
    """Simulates a resource that raises mid-execution."""
    yield {"id": 1, "value": 100}
    yield {"id": 2, "value": 200}
    # Simulate an upstream API error
    raise ConnectionError("Upstream API returned 503 Service Unavailable")


error_pipeline = dlt.pipeline(
    pipeline_name="error_demo",
    destination="duckdb",
    dataset_name="demo",
)

try:
    info = error_pipeline.run(bad_resource())
    print("Pipeline succeeded")
    print(info)

except PipelineException as e:
    # dlt-specific exception — schema errors, normalisation failures
    print(f"dlt pipeline error: {e}")

except Exception as e:
    # General exceptions — network errors, source failures, etc.
    print(f"Pipeline raised {type(e).__name__}: {e}")
    print("\nIn production: log this, trigger an alert, and decide whether to retry.")

---
## Step 7 · Inspecting `pipeline.last_trace`

After every run, dlt records a detailed trace. `pipeline.last_trace` gives you programmatic access to what happened — rows loaded, time taken, errors.

In [ ]:
# Run a clean pipeline so we have a fresh trace to inspect
trace_pipeline = dlt.pipeline(
    pipeline_name="trace_demo",
    destination="duckdb",
    dataset_name="trace",
)

@dlt.resource(write_disposition="replace")
def sample_data():
    yield [{"id": i, "value": i * 10, "label": f"item_{i}"} for i in range(1, 21)]

info = trace_pipeline.run(sample_data())
print("Run complete. Inspecting trace...\n")

# Access the last trace
trace = trace_pipeline.last_trace

print(f"Pipeline name:     {trace_pipeline.pipeline_name}")
print(f"Last run status:   {info.has_failed_jobs}")
print(f"Trace resolved at: {trace.resolved_at}")

# Normalisation info: how many records were processed
normalize_info = trace.last_normalize_info
if normalize_info:
    print(f"\nNormalisation info:")
    print(f"  row_counts: {normalize_info.row_counts}")
    print(f"  job_metrics: {normalize_info.job_metrics}")

# Load info: what was written to the destination
load_info = trace.last_load_info
if load_info:
    print(f"\nLoad info:")
    print(f"  load_id: {load_info.load_id}")
    print(f"  started_at: {load_info.started_at}")
    print(f"  finished_at: {load_info.finished_at}")
    print(f"  has_failed_jobs: {load_info.has_failed_jobs}")

# Step execution times — useful for identifying bottlenecks
print("\nStep execution times:")
for step in trace.steps:
    duration = (step.finished_at - step.started_at).total_seconds() if step.finished_at else None
    print(f"  {step.step}: {duration:.3f}s" if duration else f"  {step.step}: still running")

In [ ]:
# Helper function to extract metrics from last_trace
# Use this in production monitoring
def extract_pipeline_metrics(pipeline) -> dict:
    """Extracts key metrics from pipeline.last_trace for monitoring."""
    trace = pipeline.last_trace
    if not trace:
        return {"error": "No trace available"}

    metrics = {
        "pipeline_name": pipeline.pipeline_name,
        "has_failed_jobs": trace.last_load_info.has_failed_jobs if trace.last_load_info else None,
        "load_id": trace.last_load_info.load_id if trace.last_load_info else None,
    }

    # Row counts per table
    if trace.last_normalize_info:
        metrics["row_counts"] = trace.last_normalize_info.row_counts

    # Step durations
    step_durations = {}
    for step in trace.steps:
        if step.finished_at:
            step_durations[step.step] = round(
                (step.finished_at - step.started_at).total_seconds(), 3
            )
    metrics["step_durations_sec"] = step_durations

    return metrics


metrics = extract_pipeline_metrics(trace_pipeline)
print("Pipeline metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

**What just happened?**
- `pipeline.last_trace` is always available after a run
- `trace.last_normalize_info.row_counts` shows rows processed per table
- `trace.steps` gives you per-step timing — useful for finding bottlenecks
- `extract_pipeline_metrics()` is a pattern you'd use in production monitoring dashboards

---
## Challenge — do this yourself

1. Create a `Payment` Pydantic model with fields: `payment_id` (int), `order_id` (int), `method` (Literal: `card`, `paypal`, `crypto`), `amount` (float, must be > 0), `processed` (bool, default False)
2. Write a generator that yields a mix of 6 payments — at least 2 should fail validation
3. Run it through a dlt pipeline
4. Print how many records landed vs were skipped

In [ ]:
# Your solution here


<details>
<summary>Show solution</summary>

```python
from pydantic import BaseModel, Field
from typing import Literal

class Payment(BaseModel):
    payment_id: int
    order_id:   int
    method:     Literal["card", "paypal", "crypto"]
    amount:     float = Field(gt=0)
    processed:  bool = False

RAW_PAYMENTS = [
    {"payment_id": 1, "order_id": 101, "method": "card",    "amount": 49.99},
    {"payment_id": 2, "order_id": 102, "method": "paypal",  "amount": 99.99, "processed": True},
    {"payment_id": 3, "order_id": 103, "method": "bitcoin", "amount": 29.99},  # invalid method
    {"payment_id": 4, "order_id": 104, "method": "card",    "amount": -5.00}, # negative amount
    {"payment_id": 5, "order_id": 105, "method": "crypto",  "amount": 199.99},
    {"payment_id": 6, "order_id": 106, "method": "card",    "amount": 0.00},  # zero amount
]

@dlt.resource(primary_key="payment_id", write_disposition="replace")
def payments():
    loaded = 0
    skipped = 0
    for raw in RAW_PAYMENTS:
        try:
            p = Payment.model_validate(raw)
            loaded += 1
            yield p.model_dump()
        except ValidationError as e:
            print(f"SKIP payment_id={raw.get('payment_id')}: {[err['msg'] for err in e.errors()]}")
            skipped += 1
    print(f"Done: {loaded} loaded, {skipped} skipped")

p = dlt.pipeline(pipeline_name="payments", destination="duckdb", dataset_name="fin")
p.run(payments())
```
</details>

---
## Day 7 key concepts recap

| Concept | What to remember |
|---|---|
| Default behaviour | dlt coerces strings to numbers, NULLs on None — permissive |
| `schema_contract="freeze"` | Any schema change raises an exception |
| `schema_contract="discard_rows"` | Rows that would change schema are silently dropped |
| Generator filtering | Validate and skip before dlt sees data — cleanest approach |
| Pydantic validation | Define schema once, get automatic type coercion + validation |
| `try/except PipelineException` | Catch dlt-specific errors in production |
| `pipeline.last_trace` | Post-run metrics: rows, timing, failure status |
| `last_normalize_info.row_counts` | How many rows were processed per table |

> **Tip:** Layer your defences. Pydantic validates shape. Generator logic filters business rules. Schema contracts prevent drift. `last_trace` tells you what happened after.

---
## What's next → Day 8

**Day 8** → Production deployment: GitHub Actions cron, secrets management, Airflow integration, and structured monitoring with `last_trace`.

Mark Day 7 complete in your [tracker](../index.html).